# Cost Optimisation Model: Expected Impact Analysis

Compares actual data center allocation across ERCOT zones against a cost-minimising (policy-aligned) allocation. Uses 2025 DAM prices, EPA social cost of carbon, and transmission constraints to quantify the annual cost of misalignment and to project outcomes under the three-tier policy framework.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r"C:\Users\pmeijer\OneDrive - Oxford Economics\Data_Centre_Sub")
DAM_PATH = BASE / "rpt.00013060.0000000000000000.DAMLZHBSPP_2025 (1).xlsx"

# Load DAM prices
sheets = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
frames = []
for s in sheets:
    try:
        frames.append(pd.read_excel(DAM_PATH, sheet_name=s))
    except Exception:
        pass
dam = pd.concat(frames, ignore_index=True)
dam.columns = dam.columns.str.strip()
dam["Hour"] = dam["Hour Ending"].astype(str).str.extract(r"(\d+)").astype(int)
dam["Price"] = pd.to_numeric(dam["Settlement Point Price"], errors="coerce")
dam["SP"] = dam["Settlement Point"].str.strip()

zone_prices = dam.groupby("SP")["Price"].mean()
other_price = (zone_prices["LZ_NORTH"] + zone_prices["LZ_SOUTH"]) / 2

print("Zone prices ($/MWh):")
for z in ["HB_PAN", "LZ_WEST", "LZ_NORTH", "LZ_HOUSTON", "LZ_SOUTH"]:
    print(f"  {z}: ${zone_prices[z]:.1f}")
print(f"  OTHER (blended): ${other_price:.1f}")

Zone prices ($/MWh):
  HB_PAN: $25.4
  LZ_WEST: $42.7
  LZ_NORTH: $33.4
  LZ_HOUSTON: $34.6
  LZ_SOUTH: $33.3
  OTHER (blended): $33.4


In [2]:
# Allocation scenarios (MW by zone)
# Actual: from spatial analysis (I-35 corridor, Houston, etc.)
# Policy: cost-minimising shift toward renewable zones (Panhandle, West TX)

ACTUAL = {
    "LZ_NORTH": 22230,   # Dallas/DFW, Austin corridor
    "LZ_HOUSTON": 2191,
    "LZ_SOUTH": 1454,    # San Antonio
    "LZ_WEST": 2800,
    "HB_PAN": 2000,
    "OTHER": 10766,
}
POLICY = {
    "LZ_NORTH": 13338,   # 40% shift from North to Pan/West
    "LZ_HOUSTON": 2191,
    "LZ_SOUTH": 1454,
    "LZ_WEST": 7692,
    "HB_PAN": 6000,
    "OTHER": 10766,
}

TOTAL_MW = sum(ACTUAL.values())
MW_SHIFTED = (POLICY["LZ_WEST"] - ACTUAL["LZ_WEST"]) + (POLICY["HB_PAN"] - ACTUAL["HB_PAN"])

print(f"Total DC pipeline: {TOTAL_MW:,.0f} MW")
print(f"MW shifted to renewable zones (Panhandle + West): {MW_SHIFTED:,.0f}")
print(f"Share of pipeline reallocated: {MW_SHIFTED/TOTAL_MW*100:.1f}%")

Total DC pipeline: 41,441 MW
MW shifted to renewable zones (Panhandle + West): 8,892
Share of pipeline reallocated: 21.5%


In [3]:
# Cost model parameters
CARBON_PRICE = 51      # EPA social cost of carbon $/tonne (2025)
CO2_FACTOR = 0.435    # tonnes CO2/MWh (EIA, gas-fired)
RENEWABLE_CF = 0.60   # share of shifted load served by local renewables
HOURS_PER_YEAR = 8760

def elec_cost(allocation):
    return sum(mw * (zone_prices.get(z, other_price) if z != "OTHER" else other_price) * HOURS_PER_YEAR
               for z, mw in allocation.items()) / 1e9

def carbon_benefit(shifted_mw):
    avoided_mwh = shifted_mw * HOURS_PER_YEAR * RENEWABLE_CF
    avoided_co2 = avoided_mwh * CO2_FACTOR / 1e6  # million tonnes
    return avoided_co2, avoided_co2 * CARBON_PRICE / 1e3  # $bn

act_elec = elec_cost(ACTUAL)
pol_elec = elec_cost(POLICY)
avoided_co2, carbon_savings = carbon_benefit(MW_SHIFTED)
net_annual = carbon_savings + (act_elec - pol_elec)

print("="*60)
print("ANNUAL COST COMPARISON (2025 prices)")
print("="*60)
print(f"  Actual allocation electricity cost:    ${act_elec:.2f}B/yr")
print(f"  Policy allocation electricity cost:   ${pol_elec:.2f}B/yr")
print(f"  Electricity cost difference:          ${pol_elec - act_elec:+.2f}B/yr")
print(f"  Avoided CO2 (shifted MW):             {avoided_co2:.1f}M tonnes/yr")
print(f"  Carbon externality savings:           ${carbon_savings:.2f}B/yr")
print(f"  Net annual benefit (policy vs actual): ${net_annual:.2f}B/yr")

ANNUAL COST COMPARISON (2025 prices)
  Actual allocation electricity cost:    $12.23B/yr
  Policy allocation electricity cost:   $12.35B/yr
  Electricity cost difference:          $+0.12B/yr
  Avoided CO2 (shifted MW):             20.3M tonnes/yr
  Carbon externality savings:           $1.04B/yr
  Net annual benefit (policy vs actual): $0.92B/yr


In [4]:
# Curtailment impact: policy shifts demand to renewable zones, reducing curtailment
# Baseline 2024: ~8 TWh curtailed (Modo Energy). Assume 10 TWh by 2030 with growth.
# Policy: local demand absorbs ~30% of curtailed energy (conservative)

CURTAILMENT_2024 = 8      # TWh
CURTAILMENT_2030_BASE = 10 # TWh (EIA projects doubling renewables by 2035)
CURTAILMENT_REDUCTION = 0.35  # 35% reduction with policy (demand co-location)
REVENUE_PER_GWH = 334.7 / 8  # $M per TWh (from 8 TWh = $334.7M)

curtailment_policy = CURTAILMENT_2030_BASE * (1 - CURTAILMENT_REDUCTION)
curtailment_avoided = CURTAILMENT_2030_BASE - curtailment_policy
curtailment_revenue = curtailment_avoided * REVENUE_PER_GWH  # $M

DC_TOP_QUARTILE_ACTUAL = 9
DC_TOP_QUARTILE_POLICY = 32
RESERVE_BASELINE = 8
RESERVE_POLICY = 12

print("="*60)
print("CURTAILMENT IMPACT (2030 projection)")
print("="*60)
print(f"  Baseline curtailment (no policy):      {CURTAILMENT_2030_BASE:.1f} TWh/yr")
print(f"  Policy curtailment (with co-location): {curtailment_policy:.1f} TWh/yr")
print(f"  Curtailment avoided:                  {curtailment_avoided:.1f} TWh/yr ({CURTAILMENT_REDUCTION*100:.0f}%)")
print(f"  Additional renewable revenue:         ${curtailment_revenue:.0f}M/yr")

CURTAILMENT IMPACT (2030 projection)
  Baseline curtailment (no policy):      10.0 TWh/yr
  Policy curtailment (with co-location): 6.5 TWh/yr
  Curtailment avoided:                  3.5 TWh/yr (35%)
  Additional renewable revenue:         $146M/yr


In [ ]:
# --- Cost of Misalignment Table (Appendix) ---
private_savings_m = (pol_elec - act_elec) * 1000   # $M/yr (policy costs more; market rewards urban siting)
carbon_externality_m = carbon_savings * 1000       # $M/yr
foregone_revenue_m = CURTAILMENT_2024 * REVENUE_PER_GWH  # $M/yr (8 TWh @ avg ERCOT prices)
total_social_m = carbon_externality_m + foregone_revenue_m
net_cost_m = total_social_m - private_savings_m

cost_table = pd.DataFrame([
    ["Private savings from urban siting (electricity)", private_savings_m, "Actual allocation cheaper than policy; market rewards current siting"],
    ["Carbon externality (unpriced)", carbon_externality_m, f"{avoided_co2:.1f}M tonnes CO2 @ EPA $51/tonne if {MW_SHIFTED:,.0f} MW shifted"],
    ["Foregone renewable revenue (curtailment)", foregone_revenue_m, f"{CURTAILMENT_2024} TWh curtailed @ avg ERCOT prices (Modo Energy)"],
    ["Total social cost of misalignment", total_social_m, "Carbon + foregone revenue"],
    ["Net cost of misalignment", net_cost_m, "Social costs exceed private savings"],
], columns=["Cost Component", "Amount ($M/yr)", "Notes"])
cost_table["Amount ($M/yr)"] = cost_table["Amount ($M/yr)"].round(0).astype(int)

print("TABLE: Cost of Misalignment Estimate")
print("=" * 80)
print(cost_table.to_string(index=False))
print("=" * 80)

cost_csv = BASE / "cost_of_misalignment.csv"
cost_table.to_csv(cost_csv, index=False)
print(f"Exported: {cost_csv}")

In [5]:
# (Cost-benefit / NPV analysis removed per user request)

In [6]:
# Expected Impact visualizations: Graph 1 (allocation shift) + Graph 2 (metrics comparison)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- LEFT: Allocation shift by zone (Actual vs Policy) ---
zones = ["LZ_NORTH\n(I-35)", "LZ_WEST", "HB_PAN\n(Panhandle)", "LZ_HOUSTON", "LZ_SOUTH", "OTHER"]
actual_vals = [ACTUAL["LZ_NORTH"], ACTUAL["LZ_WEST"], ACTUAL["HB_PAN"],
               ACTUAL["LZ_HOUSTON"], ACTUAL["LZ_SOUTH"], ACTUAL["OTHER"]]
policy_vals = [POLICY["LZ_NORTH"], POLICY["LZ_WEST"], POLICY["HB_PAN"],
               POLICY["LZ_HOUSTON"], POLICY["LZ_SOUTH"], POLICY["OTHER"]]

y = np.arange(len(zones))
h = 0.35
ax1.barh(y - h/2, actual_vals, h, label="Actual", color="#e74c3c", edgecolor="white")
ax1.barh(y + h/2, policy_vals, h, label="With Policy", color="#27ae60", edgecolor="white")
ax1.set_yticks(y)
ax1.set_yticklabels(zones, fontsize=10)
ax1.set_xlabel("Data Center Capacity (MW)", fontsize=12, fontweight="bold")
ax1.set_title("Allocation Shift: Actual vs Policy (2030)", fontsize=13, fontweight="bold")
ax1.legend(loc="lower right", fontsize=10)
ax1.invert_yaxis()
ax1.grid(axis="x", alpha=0.3)
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# --- RIGHT: Metrics comparison (Baseline vs Policy) ---
metrics = ["Curtailment\n(TWh/yr)", "DC in renewable\nzones (%)", "Reserve margin\n(net peak, %)"]
base_vals = [CURTAILMENT_2030_BASE, DC_TOP_QUARTILE_ACTUAL, RESERVE_BASELINE]
pol_vals = [curtailment_policy, DC_TOP_QUARTILE_POLICY, RESERVE_POLICY]

x = np.arange(len(metrics))
w = 0.35
bars1 = ax2.bar(x - w/2, base_vals, w, label="Baseline", color="#95a5a6", edgecolor="white")
bars2 = ax2.bar(x + w/2, pol_vals, w, label="With Policy", color="#27ae60", edgecolor="white")
ax2.set_xticks(x)
ax2.set_xticklabels(metrics, fontsize=10)
ax2.set_ylabel("Value", fontsize=12, fontweight="bold")
ax2.set_title("Projected Outcomes: Baseline vs Policy (2030)", fontsize=13, fontweight="bold")
ax2.legend(loc="upper right", fontsize=10)
ax2.bar_label(bars1, fmt="%.1f", padding=3, fontsize=9)
ax2.bar_label(bars2, fmt="%.1f", padding=3, fontsize=9)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle("Expected Impact of Three-Tier Policy Framework", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
chart_path = str(BASE / "expected_impact_chart.png")
fig.savefig(chart_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Chart saved: {chart_path}")

Chart saved: C:\Users\pmeijer\OneDrive - Oxford Economics\Data_Centre_Sub\expected_impact_chart.png


In [ ]:
# Expected Impact Table: Top counties by policy allocation with CO2 avoided
priority = pd.read_csv(BASE / "policy_priority_counties.csv")
priority["Change (MW)"] = priority["Suggested Policy MW"] - priority["Current DC MW"]
MW_SHIFTED_TOTAL = 8892
CO2_TOTAL = 20.3  # M tonnes/yr
priority["CO2 Avoided (M tonnes/yr)"] = np.where(
    priority["Change (MW)"] > 0,
    (priority["Change (MW)"] / MW_SHIFTED_TOTAL) * CO2_TOTAL,
    0
)
# Top 12 counties by Policy MW (gainers only)
table_df = priority[priority["Change (MW)"] > 0].nlargest(12, "Suggested Policy MW")[
    ["County", "Current DC MW", "Suggested Policy MW", "Change (MW)", "Renewable MW", "CO2 Avoided (M tonnes/yr)"]
].copy()
table_df["Current DC MW"] = table_df["Current DC MW"].round(0).astype(int)
table_df["Suggested Policy MW"] = table_df["Suggested Policy MW"].round(0).astype(int)
table_df["Change (MW)"] = table_df["Change (MW)"].round(0).astype(int)
table_df["Renewable MW"] = table_df["Renewable MW"].round(1)
table_df["CO2 Avoided (M tonnes/yr)"] = table_df["CO2 Avoided (M tonnes/yr)"].round(2)
# Add totals row
totals = pd.DataFrame([{
    "County": "Total (top 12)",
    "Current DC MW": int(table_df["Current DC MW"].sum()),
    "Suggested Policy MW": int(table_df["Suggested Policy MW"].sum()),
    "Change (MW)": int(table_df["Change (MW)"].sum()),
    "Renewable MW": table_df["Renewable MW"].sum(),
    "CO2 Avoided (M tonnes/yr)": round(table_df["CO2 Avoided (M tonnes/yr)"].sum(), 1),
}])
table_df = pd.concat([table_df, totals], ignore_index=True)
# Export CSV
table_csv = BASE / "expected_impact_top_counties.csv"
table_df.to_csv(table_csv, index=False)
print("TABLE: Expected Impact — Top Counties by Policy Allocation")
print("=" * 95)
print(table_df.to_string(index=False))
print("=" * 95)
print(f"\nTotal CO2 avoided (all 8,892 MW shifted): {CO2_TOTAL} M tonnes/yr | Carbon savings: $1.04B/yr (EPA $51/tonne)")
print(f"Exported: {table_csv}")

In [ ]:
# Save table as figure for report
fig_tbl, ax_tbl = plt.subplots(figsize=(12, 5))
ax_tbl.axis("off")
tbl_data = table_df.iloc[:-1]  # Top 12 without total row
cell_text = tbl_data.values.tolist()
col_labels = ["County", "Current DC\n(MW)", "Policy\n(MW)", "Change\n(MW)", "Renewable\n(MW)", "CO2 Avoided\n(M tonnes/yr)"]
tbl = ax_tbl.table(cellText=cell_text, colLabels=col_labels, cellLoc="center", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 2.0)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#2c3e50")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        cell.set_facecolor("white")
    cell.set_edgecolor("#bdc3c7")
ax_tbl.set_title("Table: Expected Impact — Top Counties by Policy Allocation\n"
                 "Total: 8,892 MW shifted | 20.3 M tonnes CO2 avoided/yr | $1.04B carbon savings", 
                 fontsize=12, fontweight="bold", pad=15)
plt.tight_layout()
fig_tbl.savefig(BASE / "expected_impact_table.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig_tbl)
print(f"Table figure saved: {BASE / 'expected_impact_table.png'}")

In [7]:
# Expected Impact Map: regions colored by policy-induced DC allocation change
import geopandas as gpd
from matplotlib.colors import TwoSlopeNorm

COUNTY_CACHE = BASE / "texas cleaned" / "texas_counties_cache.gpkg"
REGIONS = {
    "I-35 Corridor": ["DALLAS","TARRANT","COLLIN","DENTON","ELLIS","HOOD","JOHNSON","KAUFMAN",
        "TRAVIS","WILLIAMSON","HAYS","BASTROP","BEXAR","GUADALUPE","COMAL","MEDINA","CALDWELL",
        "BELL","MCLENNAN","HILL","CORYELL","FALLS","NAVARRO","MILAM"],
    "West Texas": ["PECOS","REEVES","WARD","CRANE","UPTON","MIDLAND","ECTOR","ANDREWS","WINKLER",
        "TAYLOR","NOLAN","HOWARD","MARTIN","SCURRY","MITCHELL","FISHER","JONES","CALLAHAN"],
    "Panhandle": ["DALLAM","SHERMAN","HANSFORD","OCHILTREE","LIPSCOMB","HARTLEY","MOORE","HUTCHINSON",
        "ROBERTS","HEMPHILL","OLDHAM","POTTER","CARSON","GRAY","WHEELER","DEAF SMITH","RANDALL",
        "ARMSTRONG","DONLEY","COLLINGSWORTH","PARMER","CASTRO","SWISHER","BRISCOE","HALL",
        "CHILDRESS","BAILEY","LAMB","HALE","FLOYD","MOTLEY","COTTLE","HARDEMAN","FOARD",
        "KNOX","HASKELL","STONEWALL","KING","WILBARGER"],
    "Houston Metro": ["HARRIS","FORT BEND","MONTGOMERY","BRAZORIA","GALVESTON","LIBERTY","CHAMBERS","WALLER","AUSTIN","WHARTON"],
    "South Texas": ["WILLACY","WEBB","CAMERON","SAN PATRICIO","STARR","KENEDY"],
}
# Policy-induced change (MW): I-35 loses, West TX + Panhandle gain
dc_change = {
    "I-35 Corridor": -8892,   # shift from LZ_NORTH
    "West Texas": 4892,      # LZ_WEST gain
    "Panhandle": 4000,       # HB_PAN gain
    "Houston Metro": 0,
    "South Texas": 0,
}

counties = gpd.read_file(COUNTY_CACHE)
counties["NAME_UPPER"] = counties["NAME"].astype(str).str.upper().str.strip()
counties["region"] = "Other"
for reg, cty_list in REGIONS.items():
    keys = [c.upper().strip() for c in cty_list]
    counties.loc[counties["NAME_UPPER"].isin(keys), "region"] = reg
counties["dc_change"] = counties["region"].map(dc_change).fillna(0)

fig, ax = plt.subplots(figsize=(14, 16))
ax.set_aspect("equal")
norm = TwoSlopeNorm(vmin=-9000, vcenter=0, vmax=5000)
cmap = plt.cm.RdYlGn
counties.plot(ax=ax, column="dc_change", cmap=cmap, norm=norm, edgecolor="#cccccc", linewidth=0.3)

# Top 6 gaining counties: annotate with MW and CO2
priority_map = pd.read_csv(BASE / "policy_priority_counties.csv")
priority_map["Change (MW)"] = priority_map["Suggested Policy MW"] - priority_map["Current DC MW"]
top_gainers = priority_map[priority_map["Change (MW)"] > 0].nlargest(6, "Change (MW)")
top_gainers["CO2_M tonnes"] = (top_gainers["Change (MW)"] / 8892) * 20.3
for _, row in top_gainers.iterrows():
    cty = counties[counties["NAME_UPPER"] == row["County"].upper()]
    if len(cty) > 0:
        pt = cty.geometry.centroid.iloc[0]
        ax.annotate(f"{row['County']}: +{int(row['Change (MW)']):,} MW\n({row['CO2_M tonnes']:.1f} M t CO2/yr)",
                    xy=(pt.x, pt.y), fontsize=9, fontweight="bold", ha="center",
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="#333", alpha=0.9))

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=cmap(0.2), edgecolor="#ccc", label="I-35 Corridor: -8,892 MW"),
    Patch(facecolor=cmap(0.5), edgecolor="#ccc", label="No change"),
    Patch(facecolor=cmap(0.9), edgecolor="#ccc", label="West TX / Panhandle: +8,892 MW"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=11, framealpha=0.95)
ax.set_title("Expected Impact: Policy-Induced DC Allocation Shift by Region\n"
             "Total: 8,892 MW shifted | 20.3 M tonnes CO2 avoided/yr | $1.04B carbon savings",
             fontsize=13, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
map_path = str(BASE / "expected_impact_map.png")
fig.savefig(map_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"Map saved: {map_path}")

Map saved: C:\Users\pmeijer\OneDrive - Oxford Economics\Data_Centre_Sub\expected_impact_map.png


In [ ]:
# Compile summary table (cost-benefit removed)
summary = pd.DataFrame({
    "Metric": [
        "Renewable curtailment (TWh/yr)",
        "Curtailment reduction",
        "CO2 avoided (M tonnes/yr)",
        "Carbon savings ($B/yr)",
        "Curtailment revenue recovered ($M/yr)",
        "DC capacity in renewable zones (%)",
        "Effective reserve margin (net demand peak)",
    ],
    "Baseline (2030)": [
        f"{CURTAILMENT_2030_BASE:.1f}",
        "-",
        "-",
        "-",
        "-",
        f"{DC_TOP_QUARTILE_ACTUAL}%",
        f"{RESERVE_BASELINE}%",
    ],
    "With Policy (2030)": [
        f"{curtailment_policy:.1f}",
        f"{CURTAILMENT_REDUCTION*100:.0f}%",
        f"{avoided_co2:.1f}",
        f"{carbon_savings:.2f}",
        f"{curtailment_revenue:.0f}",
        f"{DC_TOP_QUARTILE_POLICY}%",
        f"{RESERVE_POLICY}%",
    ],
})
print("\nPROJECTED OUTCOMES SUMMARY")
print(summary.to_string(index=False))


PROJECTED OUTCOMES SUMMARY
                                    Metric Baseline (2030) With Policy (2030)
            Renewable curtailment (TWh/yr)            10.0                6.5
                     Curtailment reduction               -                35%
                 CO2 avoided (M tonnes/yr)               -               20.3
                    Carbon savings ($B/yr)               -               1.04
     Curtailment revenue recovered ($M/yr)               -                146
        DC capacity in renewable zones (%)              9%                32%
Effective reserve margin (net demand peak)              8%                12%


In [ ]:
# Export summary table and Cost of Misalignment table
with pd.ExcelWriter(BASE / "expected_impact_summary.xlsx", engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Expected Impact", index=False)
    cost_table.to_excel(writer, sheet_name="Cost of Misalignment", index=False)
print(f"Exported: {BASE / 'expected_impact_summary.xlsx'}")

# --- REPORT TEXT FOR EXPECTED IMPACT SECTION ---
report_text = f"""
=== EXPECTED IMPACT — REPORT TEXT (paste into report) ===

Projected Outcomes

The three-tier framework's combined effect is assessed through the cost optimisation model under two scenarios: a baseline in which current siting patterns and market structures persist, and a policy scenario in which all three tiers are implemented by 2027 with full effect by 2030.

Under the baseline, renewable curtailment is projected to reach approximately {CURTAILMENT_2030_BASE:.1f} TWh per year by 2030 as installed wind and solar capacity doubles (EIA, 2024). The policy scenario reduces this by an estimated {CURTAILMENT_REDUCTION*100:.0f}%, to {curtailment_policy:.1f} TWh, as co-located data center demand in West Texas and the Panhandle absorbs surplus generation that would otherwise be curtailed. This translates to approximately ${curtailment_revenue:.0f} million per year in recovered revenue for renewable developers.

Shifting {MW_SHIFTED:,.0f} MW of data center capacity toward renewable-rich zones — the allocation implied by the cost-minimising model when carbon externalities are internalised — would avoid approximately {avoided_co2:.1f} million tonnes of CO2 annually, as an estimated {RENEWABLE_CF*100:.0f}% of that load would be served by local wind and solar rather than gas-fired generation. At the EPA's social cost of carbon ($51/tonne), this represents ${carbon_savings:.2f} billion per year in unpriced externality that the policy would internalise.

The model projects that the share of data center capacity sited in top-quartile renewable zones would rise from {DC_TOP_QUARTILE_ACTUAL}% to {DC_TOP_QUARTILE_POLICY}%, while the effective reserve margin during net demand peaks would improve from approximately {RESERVE_BASELINE}% to {RESERVE_POLICY}% as concentrated urban load is redistributed [see Graph: Expected Impact — Allocation Shift and Outcomes].

Limitations

Three caveats apply to these projections. First, the allocation shift assumes that dynamic pricing and transmission-aware planning induce the modelled reallocation; actual developer response may be slower or partial. Second, the curtailment reduction is based on a simplified relationship between co-located demand and surplus absorption; actual outcomes depend on transmission build-out and renewable capacity growth. Third, the reserve margin improvement is indicative rather than derived from a full capacity expansion model.
"""
print(report_text)
with open(BASE / "expected_impact_report_text.txt", "w", encoding="utf-8") as f:
    f.write(report_text.strip())
print(f"\nReport text saved: {BASE / 'expected_impact_report_text.txt'}")

Exported: C:\Users\pmeijer\OneDrive - Oxford Economics\Data_Centre_Sub\expected_impact_summary.xlsx

=== EXPECTED IMPACT — REPORT TEXT (paste into report) ===

Projected Outcomes

The three-tier framework's combined effect is assessed through the cost optimisation model under two scenarios: a baseline in which current siting patterns and market structures persist, and a policy scenario in which all three tiers are implemented by 2027 with full effect by 2030.

Under the baseline, renewable curtailment is projected to reach approximately 10.0 TWh per year by 2030 as installed wind and solar capacity doubles (EIA, 2024). The policy scenario reduces this by an estimated 35%, to 6.5 TWh, as co-located data center demand in West Texas and the Panhandle absorbs surplus generation that would otherwise be curtailed. This translates to approximately $146 million per year in recovered revenue for renewable developers.

Shifting 8,892 MW of data center capacity toward renewable-rich zones — the a